# Yahoo Finance Market Data Extraction and Train-Only Feature Audit

**Use this notebook with the latest revised Phase 0–4 training notebooks (`v3_Training_Dedup`).**

This notebook generates the exact market files expected by those five notebooks:

- `/content/drive/MyDrive/Crypto_Research/data/market/BTC_market.csv`
- `/content/drive/MyDrive/Crypto_Research/data/market/ETH_market.csv`
- `/content/drive/MyDrive/Crypto_Research/data/market/selected_market_features.json`
- `/content/drive/MyDrive/Crypto_Research/data/market/market_feature_vif_history.csv`
- `/content/drive/MyDrive/Crypto_Research/data/market/market_feature_cv_score_folds.csv`
- `/content/drive/MyDrive/Crypto_Research/data/market/market_feature_cv_score_summary.csv`

**Data source:** Yahoo Finance through `yfinance` for all market series:

- BTC: `BTC-USD`
- ETH: `ETH-USD`
- S&P 500: `^GSPC`
- VIX: `^VIX`

The methodology is aligned with the latest Phase notebooks:

- Raw market data begin on **2022-04-01** for the warm-up month.
- The formal forecasting sample begins on **2022-05-01**.
- The prediction target is the **next-day log return**.
- Raw price levels are retained only for audit/feature engineering, not as candidate model predictors.
- Candidate predictors use returns, ratios, volatility, and changes.
- Feature screening uses the **training period only**, defined by forecast target date.
- Iterative VIF screening uses threshold **5.0** and produces one common BTC/ETH market feature set.
- Time-series CV drop-column importance is saved as a diagnostic only and does not determine inclusion.

After this notebook finishes successfully, run the latest notebooks in order: **Phase 0 → Phase 1 → Phase 2 → Phase 3 → Phase 4**.


In [1]:
!pip -q install -U yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 6.6 MB/s eta 0:00:00


In [2]:
# ============================================================
# 1. IMPORTS AND SETTINGS
# ============================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")

# Mount Google Drive in Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# Change PROJECT_DIR only if your project is stored elsewhere.
PROJECT_DIR = Path('/content/drive/MyDrive/Crypto_Research')
MARKET_DIR = PROJECT_DIR / 'data' / 'market'
MARKET_DIR.mkdir(parents=True, exist_ok=True)

RAW_START_DATE = pd.Timestamp('2022-04-01')
DATA_END_DATE = pd.Timestamp('2026-08-31')
# yfinance uses an exclusive end date.
YAHOO_END_EXCLUSIVE = DATA_END_DATE + pd.Timedelta(days=1)

FORMAL_START_DATE = pd.Timestamp('2022-05-01')
TRAIN_END_DATE = pd.Timestamp('2025-03-31')

VIF_THRESHOLD = 5.0
CV_SPLITS = 5
RANDOM_STATE = 42

YAHOO_TICKERS = {
    'BTC': 'BTC-USD',
    'ETH': 'ETH-USD',
    'SP500': '^GSPC',
    'VIX': '^VIX',
}

# Stationary / scale-stable candidate predictors used by all latest phases.
MARKET_CANDIDATE_FEATURES = [
    'Log_Return',
    'Return_lag1',
    'Return_lag3',
    'Return_lag5',
    'MA20_Distance',
    'RSI',
    'volatility_20d',
    'volume_change',
    'volume_vs_MA5',
    'intraday_range',
    'SP500_Return',
    'VIX_Change',
]

print('Market output directory:', MARKET_DIR)
print('Yahoo download period:', RAW_START_DATE.date(), 'to', DATA_END_DATE.date())


Mounted at /content/drive
Market output directory: /content/drive/MyDrive/Crypto_Research/data/market
Yahoo download period: 2022-04-01 to 2026-08-31


In [3]:
# ============================================================
# 2. YAHOO FINANCE DOWNLOAD HELPERS
# ============================================================

def _flatten_yfinance_columns(frame):
    """Flatten yfinance MultiIndex columns for a single ticker if necessary."""
    df = frame.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    return df


def fetch_yahoo_daily(ticker, require_volume=True):
    """Download one daily Yahoo Finance series and standardize OHLCV columns."""
    df = yf.download(
        ticker,
        start=RAW_START_DATE.strftime('%Y-%m-%d'),
        end=YAHOO_END_EXCLUSIVE.strftime('%Y-%m-%d'),
        interval='1d',
        auto_adjust=False,
        actions=False,
        progress=False,
        threads=False,
    )

    if df is None or df.empty:
        raise RuntimeError(f'{ticker}: Yahoo Finance returned no data.')

    df = _flatten_yfinance_columns(df).reset_index()

    date_col = next((c for c in df.columns if str(c).lower() in {'date', 'datetime'}), None)
    if date_col is None:
        raise ValueError(f'{ticker}: no Date column found. Returned columns: {list(df.columns)}')
    if date_col != 'Date':
        df = df.rename(columns={date_col: 'Date'})

    required = ['Date', 'Open', 'High', 'Low', 'Close']
    if require_volume:
        required.append('Volume')

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'{ticker}: missing expected columns {missing}. Returned columns: {list(df.columns)}')

    df = df[required].copy()
    df['Date'] = (
        pd.to_datetime(df['Date'], errors='coerce', utc=True)
        .dt.tz_convert(None)
        .dt.normalize()
    )

    for c in [c for c in required if c != 'Date']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    return (
        df.dropna(subset=['Date', 'Close'])
        .sort_values('Date')
        .drop_duplicates('Date', keep='last')
        .reset_index(drop=True)
    )


def index_close_frame(ticker):
    """Download an index and retain only Date and Close for market controls."""
    df = fetch_yahoo_daily(ticker, require_volume=False)
    return df[['Date', 'Close']].copy()


In [4]:
# ============================================================
# 3. DOWNLOAD BTC, ETH, S&P 500, AND VIX FROM YAHOO FINANCE
# ============================================================

btc_raw = fetch_yahoo_daily(YAHOO_TICKERS['BTC'], require_volume=True)
print('BTC downloaded:', len(btc_raw), 'rows')

eth_raw = fetch_yahoo_daily(YAHOO_TICKERS['ETH'], require_volume=True)
print('ETH downloaded:', len(eth_raw), 'rows')

spx_raw = index_close_frame(YAHOO_TICKERS['SP500'])
print('S&P 500 downloaded:', len(spx_raw), 'rows')

vix_raw = index_close_frame(YAHOO_TICKERS['VIX'])
print('VIX downloaded:', len(vix_raw), 'rows')

print('\nDownloaded date ranges:')
for name, frame in {
    'BTC': btc_raw,
    'ETH': eth_raw,
    'S&P 500': spx_raw,
    'VIX': vix_raw,
}.items():
    print(f'{name}: {frame["Date"].min().date()} to {frame["Date"].max().date()}')


BTC downloaded: 1614 rows
ETH downloaded: 1614 rows
S&P 500 downloaded: 1107 rows
VIX downloaded: 1108 rows

Downloaded date ranges:
BTC: 2022-04-01 to 2026-08-31
ETH: 2022-04-01 to 2026-08-31
S&P 500: 2022-04-01 to 2026-08-31
VIX: 2022-04-01 to 2026-08-31


In [5]:
# ============================================================
# 4. FEATURE ENGINEERING
# ============================================================

def calculate_rsi(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1/window, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1/window, adjust=False, min_periods=window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(50.0)


def build_market_file(asset_df, spx_df, vix_df):
    """Create a crypto-calendar market file with raw fields plus stationary predictors."""
    df = asset_df.copy()
    df = df[(df['Date'] >= RAW_START_DATE) & (df['Date'] <= DATA_END_DATE)].copy()

    # Merge Yahoo index closes onto the 7-day crypto calendar.
    spx = spx_df.rename(columns={'Close': 'SP500Close'})
    vix = vix_df.rename(columns={'Close': 'VIXClose'})
    df = df.merge(spx[['Date', 'SP500Close']], on='Date', how='left')
    df = df.merge(vix[['Date', 'VIXClose']], on='Date', how='left')

    # Forward fill only market-wide series across weekends/holidays.
    # No backward fill is used, so future information is never introduced.
    df['SP500Close'] = df['SP500Close'].ffill()
    df['VIXClose'] = df['VIXClose'].ffill()

    # Raw audit variables
    df['Daily_Return'] = df['Close'].pct_change()
    df['Log_Return'] = np.log(df['Close']).diff()
    df['MA_20'] = df['Close'].rolling(20, min_periods=20).mean()

    # Stationary / scale-stable modeling variables
    df['Return_lag1'] = df['Log_Return'].shift(1)
    df['Return_lag3'] = df['Log_Return'].shift(3)
    df['Return_lag5'] = df['Log_Return'].shift(5)
    df['MA20_Distance'] = df['Close'] / df['MA_20'] - 1.0
    df['RSI'] = calculate_rsi(df['Close'], 14)
    df['volatility_20d'] = df['Log_Return'].rolling(20, min_periods=20).std()
    df['volume_change'] = np.log1p(df['Volume']).diff()
    df['volume_vs_MA5'] = df['Volume'] / df['Volume'].rolling(5, min_periods=5).mean()
    df['intraday_range'] = np.log(df['High'] / df['Low'])
    df['SP500_Return'] = np.log(df['SP500Close']).diff().fillna(0.0)
    df['VIX_Change'] = np.log(df['VIXClose']).diff().fillna(0.0)

    # Target-date fields used only for auditing / split definition.
    df['Target_Log_Return'] = np.log(df['Close'].shift(-1) / df['Close'])
    df['Target_Date'] = df['Date'].shift(-1)

    df = df.replace([np.inf, -np.inf], np.nan)
    return df.reset_index(drop=True)


btc_market = build_market_file(btc_raw, spx_raw, vix_raw)
eth_market = build_market_file(eth_raw, spx_raw, vix_raw)

print('BTC market period:', btc_market['Date'].min().date(), 'to', btc_market['Date'].max().date())
print('ETH market period:', eth_market['Date'].min().date(), 'to', eth_market['Date'].max().date())


BTC market period: 2022-04-01 to 2026-08-31
ETH market period: 2022-04-01 to 2026-08-31


In [6]:
# ============================================================
# 5. TRAIN-ONLY COMMON VIF FEATURE SCREENING
# ============================================================

def training_rows(df):
    """Select rows by forecast target date, matching the revised experiment split."""
    mask = (
        (df['Target_Date'] >= FORMAL_START_DATE) &
        (df['Target_Date'] <= TRAIN_END_DATE)
    )
    return df.loc[mask].copy()


def vif_for_asset(df, features):
    x = df[list(features)].apply(pd.to_numeric, errors='coerce')
    x = x.replace([np.inf, -np.inf], np.nan).dropna()
    x = x.loc[:, x.nunique() > 1]
    if x.shape[1] < 2:
        return {c: np.nan for c in features}

    scaled = StandardScaler().fit_transform(x)
    out = {}
    for i, col in enumerate(x.columns):
        try:
            out[col] = float(variance_inflation_factor(scaled, i))
        except Exception:
            out[col] = np.inf
    return out


train_frames = {
    'BTC': training_rows(btc_market),
    'ETH': training_rows(eth_market),
}

selected = MARKET_CANDIDATE_FEATURES.copy()
vif_history = []
iteration = 0

while len(selected) > 1:
    iteration += 1
    asset_vifs = {
        asset: vif_for_asset(frame, selected)
        for asset, frame in train_frames.items()
    }

    max_vifs = {}
    for feature in selected:
        values = [asset_vifs[a].get(feature, np.nan) for a in asset_vifs]
        finite = [v for v in values if pd.notna(v)]
        max_vifs[feature] = max(finite) if finite else np.inf

    worst_feature = max(max_vifs, key=max_vifs.get)
    worst_vif = max_vifs[worst_feature]

    for feature in selected:
        vif_history.append({
            'iteration': iteration,
            'feature': feature,
            'BTC_VIF': asset_vifs['BTC'].get(feature, np.nan),
            'ETH_VIF': asset_vifs['ETH'].get(feature, np.nan),
            'Max_VIF': max_vifs[feature],
            'removed_this_iteration': feature == worst_feature and worst_vif > VIF_THRESHOLD,
        })

    if worst_vif <= VIF_THRESHOLD:
        break

    print(f'Removing {worst_feature}: max training VIF = {worst_vif:.3f}')
    selected.remove(worst_feature)

vif_history_df = pd.DataFrame(vif_history)

print('\nSelected common market features:')
for feature in selected:
    print(' -', feature)

print('\nFinal maximum VIF must be <=', VIF_THRESHOLD)


Removing MA20_Distance: max training VIF = 7.302

Selected common market features:
 - Log_Return
 - Return_lag1
 - Return_lag3
 - Return_lag5
 - RSI
 - volatility_20d
 - volume_change
 - volume_vs_MA5
 - intraday_range
 - SP500_Return
 - VIX_Change

Final maximum VIF must be <= 5.0


In [7]:
# ============================================================
# 6. TIME-SERIES CV DROP-COLUMN RELEVANCE DIAGNOSTIC
# ============================================================
# This score is reported for transparency, but it does not decide inclusion.
# Positive score means that removing the feature worsened validation RMSE.


def cv_dropcolumn_scores(df, features, n_splits=5):
    working = df[list(features) + ['Target_Log_Return']].dropna().reset_index(drop=True)
    X = working[list(features)].copy()
    y = working['Target_Log_Return'].to_numpy(float)

    splitter = TimeSeriesSplit(n_splits=n_splits)
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(splitter.split(X), start=1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        full_model = HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=200,
            max_depth=3,
            l2_regularization=1.0,
            random_state=RANDOM_STATE,
        )
        full_model.fit(X_train, y_train)
        full_pred = full_model.predict(X_val)
        full_rmse = mean_squared_error(y_val, full_pred) ** 0.5

        for feature in features:
            keep = [c for c in features if c != feature]
            reduced_model = HistGradientBoostingRegressor(
                learning_rate=0.05,
                max_iter=200,
                max_depth=3,
                l2_regularization=1.0,
                random_state=RANDOM_STATE,
            )
            reduced_model.fit(X_train[keep], y_train)
            reduced_pred = reduced_model.predict(X_val[keep])
            reduced_rmse = mean_squared_error(y_val, reduced_pred) ** 0.5
            relative_increase = (reduced_rmse - full_rmse) / full_rmse if full_rmse > 0 else np.nan

            fold_rows.append({
                'fold': fold,
                'feature': feature,
                'full_rmse': full_rmse,
                'rmse_without_feature': reduced_rmse,
                'relative_rmse_increase': relative_increase,
            })

    fold_df = pd.DataFrame(fold_rows)
    summary = (
        fold_df.groupby('feature', as_index=False)
        .agg(
            mean_relative_rmse_increase=('relative_rmse_increase', 'mean'),
            std_relative_rmse_increase=('relative_rmse_increase', 'std'),
            positive_folds=('relative_rmse_increase', lambda s: int((s > 0).sum())),
            folds=('relative_rmse_increase', 'size'),
        )
        .sort_values('mean_relative_rmse_increase', ascending=False)
    )
    return fold_df, summary


score_summaries = []
score_folds = []
for asset, train_df in train_frames.items():
    folds, summary = cv_dropcolumn_scores(train_df, selected, n_splits=CV_SPLITS)
    folds.insert(0, 'asset', asset)
    summary.insert(0, 'asset', asset)
    score_folds.append(folds)
    score_summaries.append(summary)

cv_score_folds = pd.concat(score_folds, ignore_index=True)
cv_score_summary = pd.concat(score_summaries, ignore_index=True)

display(cv_score_summary.round(4))


,asset,feature,mean_relative_rmse_increase,std_relative_rmse_increase,positive_folds,folds
0,BTC,RSI,0.0102,0.0128,3,5
1,BTC,VIX_Change,0.0042,0.0094,2,5
2,BTC,Return_lag3,-0.0013,0.0068,3,5
3,BTC,Log_Return,-0.0022,0.0068,2,5
4,BTC,volume_vs_MA5,-0.0026,0.0096,3,5
5,BTC,SP500_Return,-0.0046,0.0230,2,5
6,BTC,Return_lag1,-0.0053,0.0136,2,5
7,BTC,volatility_20d,-0.0072,0.0278,3,5
8,BTC,Return_lag5,-0.0078,0.0199,2,5
9,BTC,volume_change,-0.0101,0.0087,1,5


In [8]:
# ============================================================
# 7. SAVE MARKET FILES AND AUDIT EVIDENCE
# ============================================================

# Save raw + engineered market files. The modeling notebooks use only
# the train-selected stationary feature names stored in the JSON below.
btc_path = MARKET_DIR / 'BTC_market.csv'
eth_path = MARKET_DIR / 'ETH_market.csv'

btc_market.to_csv(btc_path, index=False)
eth_market.to_csv(eth_path, index=False)

vif_history_df.to_csv(MARKET_DIR / 'market_feature_vif_history.csv', index=False)
cv_score_folds.to_csv(MARKET_DIR / 'market_feature_cv_score_folds.csv', index=False)
cv_score_summary.to_csv(MARKET_DIR / 'market_feature_cv_score_summary.csv', index=False)

selection_payload = {
    'market_data_source': 'Yahoo Finance via yfinance',
    'yahoo_tickers': YAHOO_TICKERS,
    'raw_data_period': {
        'start': str(RAW_START_DATE.date()),
        'end': str(DATA_END_DATE.date()),
    },
    'selection_period_by_target_date': {
        'start': str(FORMAL_START_DATE.date()),
        'end': str(TRAIN_END_DATE.date()),
    },
    'candidate_features': MARKET_CANDIDATE_FEATURES,
    'selection_method': 'Iterative VIF screening on training data only; common BTC/ETH set based on maximum VIF across assets.',
    'vif_threshold': VIF_THRESHOLD,
    'selected_features': selected,
    'cv_relevance_diagnostic': 'Expanding time-series CV drop-column relative RMSE increase; diagnostic only, not an inclusion rule.',
}

with open(MARKET_DIR / 'selected_market_features.json', 'w', encoding='utf-8') as f:
    json.dump(selection_payload, f, indent=2)

print('Saved:', btc_path)
print('Saved:', eth_path)
print('Saved:', MARKET_DIR / 'selected_market_features.json')
print('Saved:', MARKET_DIR / 'market_feature_vif_history.csv')
print('Saved:', MARKET_DIR / 'market_feature_cv_score_folds.csv')
print('Saved:', MARKET_DIR / 'market_feature_cv_score_summary.csv')


Saved: /content/drive/MyDrive/Crypto_Research/data/market/BTC_market.csv
Saved: /content/drive/MyDrive/Crypto_Research/data/market/ETH_market.csv
Saved: /content/drive/MyDrive/Crypto_Research/data/market/selected_market_features.json
Saved: /content/drive/MyDrive/Crypto_Research/data/market/market_feature_vif_history.csv
Saved: /content/drive/MyDrive/Crypto_Research/data/market/market_feature_cv_score_folds.csv
Saved: /content/drive/MyDrive/Crypto_Research/data/market/market_feature_cv_score_summary.csv


In [9]:
# ============================================================
# 8. FINAL QUALITY CHECK FOR LATEST PHASE 0-4 NOTEBOOKS
# ============================================================

required_output_columns = [
    'Date', 'Open', 'High', 'Low', 'Close', 'Volume',
    'SP500Close', 'VIXClose',
    'Target_Log_Return', 'Target_Date',
] + MARKET_CANDIDATE_FEATURES

for asset, df in {'BTC': btc_market, 'ETH': eth_market}.items():
    formal = df[
        (df['Target_Date'] >= FORMAL_START_DATE) &
        (df['Target_Date'] <= DATA_END_DATE)
    ].copy()

    missing_columns = [c for c in required_output_columns if c not in df.columns]
    if missing_columns:
        raise ValueError(f'{asset}: missing columns required by the latest phase notebooks: {missing_columns}')
    if formal.empty:
        raise ValueError(f'{asset}: formal modeling sample is empty.')

    print('\n' + asset)
    print('Formal target-date rows:', len(formal))
    print('Target date:', formal['Target_Date'].min().date(), 'to', formal['Target_Date'].max().date())
    print('Missing selected feature values:')
    print(formal[selected].isna().sum())

print('\nSelected features used by all five latest revised phases:')
print(selected)
print('\nReady for Phase 0 -> Phase 1 -> Phase 2 -> Phase 3 -> Phase 4.')



BTC
Formal target-date rows: 1584
Target date: 2022-05-01 to 2026-08-31
Missing selected feature values:
Log_Return        0
Return_lag1       0
Return_lag3       0
Return_lag5       0
RSI               0
volatility_20d    0
volume_change     0
volume_vs_MA5     0
intraday_range    0
SP500_Return      0
VIX_Change        0
dtype: int64

ETH
Formal target-date rows: 1584
Target date: 2022-05-01 to 2026-08-31
Missing selected feature values:
Log_Return        0
Return_lag1       0
Return_lag3       0
Return_lag5       0
RSI               0
volatility_20d    0
volume_change     0
volume_vs_MA5     0
intraday_range    0
SP500_Return      0
VIX_Change        0
dtype: int64

Selected features used by all five latest revised phases:
['Log_Return', 'Return_lag1', 'Return_lag3', 'Return_lag5', 'RSI', 'volatility_20d', 'volume_change', 'volume_vs_MA5', 'intraday_range', 'SP500_Return', 'VIX_Change']

Ready for Phase 0 -> Phase 1 -> Phase 2 -> Phase 3 -> Phase 4.
